In [7]:
# ==============================================================================
# PREREQUISITE: 
# Ensure you have added the "instacart-market-basket-analysis" dataset 
# to your Kaggle environment via the "Add Data" button.
# ==============================================================================

!pip install -q delta-spark

import os
import zipfile
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, input_file_name, when, 
    avg, count, sum as _sum, round as _round, max as _max
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, FloatType
)
from delta import configure_spark_with_delta_pip

print("1. Initializing Spark Session with Delta Lake...")
builder = (
    SparkSession.builder
    .appName("Instacart_FMCG_Lakehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.adaptive.enabled", "true")
    # Optimized memory for Kaggle's 30GB environment
    .config("spark.driver.memory", "12g")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# ==============================================================================
# DIRECTORY SETUP & UNZIP LOGIC
# ==============================================================================
print("2. Setting up directories and checking file formats...")
BASE_RAW_DIR = "/kaggle/input/datasets/psparks/instacart-market-basket-analysis"
WORKING_DATA_DIR = "/kaggle/working/data"
LAKEHOUSE_DIR = "/kaggle/working/lakehouse"

BRONZE_DIR = f"{LAKEHOUSE_DIR}/bronze"
SILVER_DIR = f"{LAKEHOUSE_DIR}/silver"
GOLD_DIR = f"{LAKEHOUSE_DIR}/gold"

os.makedirs(WORKING_DATA_DIR, exist_ok=True)

def get_csv_path(filename):
    """Kaggle sometimes stores files as .csv.zip. This unzips them if needed."""
    zip_path = f"{BASE_RAW_DIR}/{filename}.zip"
    csv_input_path = f"{BASE_RAW_DIR}/{filename}"
    extracted_path = f"{WORKING_DATA_DIR}/{filename}"
    
    if os.path.exists(csv_input_path):
        return csv_input_path
    elif os.path.exists(zip_path):
        if not os.path.exists(extracted_path):
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extract(filename, WORKING_DATA_DIR)
        return extracted_path
    else:
        # Fallback to PySpark wildcard if exact path mapping fails
        return f"{BASE_RAW_DIR}/{filename}*"

# ==============================================================================
# BRONZE LAYER (RAW INGESTION)
# ==============================================================================
print("3. Building Bronze Layer (Raw Ingestion & Audit Logs)...")
schemas = {
    "departments.csv": StructType([
        StructField("department_id", IntegerType(), False),
        StructField("department", StringType(), True)
    ]),
    "aisles.csv": StructType([
        StructField("aisle_id", IntegerType(), False),
        StructField("aisle", StringType(), True)
    ]),
    "products.csv": StructType([
        StructField("product_id", IntegerType(), False),
        StructField("product_name", StringType(), True),
        StructField("aisle_id", IntegerType(), True),
        StructField("department_id", IntegerType(), True)
    ]),
    "orders.csv": StructType([
        StructField("order_id", IntegerType(), False),
        StructField("user_id", IntegerType(), False),
        StructField("eval_set", StringType(), True),
        StructField("order_number", IntegerType(), True),
        StructField("order_dow", IntegerType(), True),
        StructField("order_hour_of_day", IntegerType(), True),
        StructField("days_since_prior_order", FloatType(), True)
    ]),
    "order_products__prior.csv": StructType([
        StructField("order_id", IntegerType(), False),
        StructField("product_id", IntegerType(), False),
        StructField("add_to_cart_order", IntegerType(), True),
        StructField("reordered", IntegerType(), True)
    ]),
    "order_products__train.csv": StructType([
        StructField("order_id", IntegerType(), False),
        StructField("product_id", IntegerType(), False),
        StructField("add_to_cart_order", IntegerType(), True),
        StructField("reordered", IntegerType(), True)
    ])
}

def ingest_to_bronze(filename, schema):
    table_name = filename.replace(".csv", "").replace("__", "_")
    input_path = get_csv_path(filename)
    output_path = f"{BRONZE_DIR}/{table_name}"
    
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .schema(schema)
        .load(input_path)
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", input_file_name())
    )
    df.write.format("delta").mode("overwrite").save(output_path)
    print(f"   -> [Bronze] Loaded {table_name}: {df.count():,} rows")

for file, schema in schemas.items():
    ingest_to_bronze(file, schema)

# ==============================================================================
# SILVER LAYER (CLEANING & CONSOLIDATING)
# ==============================================================================
print("4. Building Silver Layer (Data Cleansing & Denormalization)...")
b_orders = spark.read.format("delta").load(f"{BRONZE_DIR}/orders")
b_prior = spark.read.format("delta").load(f"{BRONZE_DIR}/order_products_prior")
b_train = spark.read.format("delta").load(f"{BRONZE_DIR}/order_products_train")
b_products = spark.read.format("delta").load(f"{BRONZE_DIR}/products")
b_departments = spark.read.format("delta").load(f"{BRONZE_DIR}/departments")
b_aisles = spark.read.format("delta").load(f"{BRONZE_DIR}/aisles")

# Silver Products (Denormalized) - FIXED AMBIGUOUS REFERENCE
s_products = (
    b_products
    .join(b_departments, "department_id", "left")
    .join(b_aisles, "aisle_id", "left")
    .select(
        b_products["product_id"], 
        b_products["product_name"], 
        b_products["department_id"], 
        b_departments["department"], 
        b_products["aisle_id"], 
        b_aisles["aisle"], 
        b_products["_ingested_at"] # Explicitly selecting from b_products
    )
)
s_products.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/dim_products")

# Silver Orders (Clean Nulls)
s_orders = (
    b_orders
    .withColumn("days_since_prior_order", when(col("days_since_prior_order").isNull(), 0.0).otherwise(col("days_since_prior_order")))
    .withColumn("is_first_order", when(col("order_number") == 1, 1).otherwise(0))
    .select("order_id", "user_id", "eval_set", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order", "is_first_order")
)
s_orders.write.format("delta").mode("overwrite").partitionBy("order_dow").save(f"{SILVER_DIR}/fact_orders")

# Silver Order Items (Union Train + Prior into one massive table)
s_items = (
    b_prior.select("order_id", "product_id", "add_to_cart_order", "reordered")
    .unionByName(b_train.select("order_id", "product_id", "add_to_cart_order", "reordered"))
)
s_items.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/fact_order_items")
print("   -> [Silver] Dimensions and Facts saved successfully.")
# ==============================================================================
# GOLD LAYER (BUSINESS AGGREGATIONS)
# ==============================================================================
print("5. Building Gold Layer (Analytical Marts for BI)...")
f_orders = spark.read.format("delta").load(f"{SILVER_DIR}/fact_orders")
f_items = spark.read.format("delta").load(f"{SILVER_DIR}/fact_order_items")
d_products = spark.read.format("delta").load(f"{SILVER_DIR}/dim_products")

# Core unified table for analytics
enriched = f_items.join(f_orders, "order_id", "inner").join(d_products, "product_id", "inner")

# Mart 1: Department Performance
g_dept = (
    enriched
    .groupBy("department_id", "department")
    .agg(
        count("order_id").alias("total_items_sold"),
        _sum("reordered").alias("total_reordered_items"),
        _round((_sum("reordered") / count("order_id")) * 100, 2).alias("reorder_rate_pct"),
        _round(avg("add_to_cart_order"), 2).alias("avg_cart_add_position")
    )
    .orderBy(col("total_items_sold").desc())
)
g_dept.write.format("delta").mode("overwrite").save(f"{GOLD_DIR}/department_metrics")

# Mart 2: Customer Profile
g_cust = (
    f_orders
    .groupBy("user_id")
    .agg(
        _max("order_number").alias("lifetime_orders"),
        _round(avg("days_since_prior_order"), 1).alias("avg_days_between_orders"),
        _round(avg("order_hour_of_day"), 0).alias("preferred_order_hour")
    )
)
g_cust.write.format("delta").mode("overwrite").save(f"{GOLD_DIR}/customer_profiles")
print("   -> [Gold] Analytical marts saved successfully.")

# ==============================================================================
# VERIFICATION & DISPLAY
# ==============================================================================
print("\n=== PIPELINE EXECUTION COMPLETE ===")
print("\n[Preview] Top 5 Departments by Total Items Sold:")
spark.read.format("delta").load(f"{GOLD_DIR}/department_metrics").show(5, truncate=False)

print("\n[Preview] Top 5 Most Active Customers:")
spark.read.format("delta").load(f"{GOLD_DIR}/customer_profiles").orderBy(col("lifetime_orders").desc()).show(5, truncate=False)

1. Initializing Spark Session with Delta Lake...
2. Setting up directories and checking file formats...
3. Building Bronze Layer (Raw Ingestion & Audit Logs)...


   -> [Bronze] Loaded departments: 21 rows


   -> [Bronze] Loaded aisles: 134 rows


   -> [Bronze] Loaded products: 49,688 rows


   -> [Bronze] Loaded orders: 3,421,083 rows


...

   -> [Bronze] Loaded order_products_train: 1,384,617 rows
4. Building Silver Layer (Data Cleansing & Denormalization)...


   -> [Silver] Dimensions and Facts saved successfully.
5. Building Gold Layer (Analytical Marts for BI)...


   -> [Gold] Analytical marts saved successfully.

=== PIPELINE EXECUTION COMPLETE ===

[Preview] Top 5 Departments by Total Items Sold:


+-------------+----------+----------------+---------------------+----------------+---------------------+
|department_id|department|total_items_sold|total_reordered_items|reorder_rate_pct|avg_cart_add_position|
+-------------+----------+----------------+---------------------+----------------+---------------------+
|4            |produce   |9888378         |6432596              |65.05           |8.04                 |
|16           |dairy eggs|5631067         |3773723              |67.02           |7.51                 |
|19           |snacks    |3006412         |1727075              |57.45           |9.2                  |
|7            |beverages |2804175         |1832952              |65.37           |6.98                 |
|1            |frozen    |2336858         |1268058              |54.26           |9.02                 |
+-------------+----------+----------------+---------------------+----------------+---------------------+
only showing top 5 rows

[Preview] Top 5 Most Active Cu

+-------+---------------+-----------------------+--------------------+
|user_id|lifetime_orders|avg_days_between_orders|preferred_order_hour|
+-------+---------------+-----------------------+--------------------+
|87238  |100            |3.1                    |15.0                |
|34014  |100            |3.4                    |13.0                |
|167316 |100            |2.9                    |11.0                |
|15896  |100            |1.9                    |16.0                |
|53001  |100            |2.2                    |14.0                |
+-------+---------------+-----------------------+--------------------+
only showing top 5 rows
